In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("LoadFactAffiliatePerformance").getOrCreate()

# ==========================================================
# 0. LOAD DIMENSIONS
# ==========================================================
dim_operator_df = (
    spark.read.table("LH_PeakMedia.dbo.dim_operator")
    .select(
        F.col("operator_id").cast("int").alias("operator_id"),
        F.trim(F.col("operator_name")).alias("operator_name")
    )
)

dim_currency_df = (
    spark.read.table("LH_PeakMedia.dbo.dim_currency")
    .select(
        F.col("currency_id").cast("int").alias("currency_id"),
        F.trim(F.col("currency")).alias("currency")
    )
)

# ==========================================================
# 1. LOAD TRACKAFF DATA
# ==========================================================
trackaff_src = spark.read.table("LH_PeakMedia.dbo.trackaff_clean")

trackaff_df = (
    trackaff_src
    .select(
        F.to_date(F.col("date"), "yyyy-MM-dd").alias("date"),
        F.col("tracking_id").cast("string").alias("tracking_id"),
        F.trim(F.col("operator_name")).alias("operator_name"),
        F.col("registrations").cast("int").alias("registrations"),
        F.col("ftds").cast("int").alias("ftds"),
        F.col("total_commission").cast("decimal(18,2)").alias("total_commission"),
        F.trim(F.col("currency")).alias("currency"),
        F.substring(F.lpad(F.col("tracking_id").cast("string"), 12, "0"), 7, 3).cast("int").alias("card_id"),
        F.substring(F.lpad(F.col("tracking_id").cast("string"), 12, "0"), 10, 3).alias("campaign_id")
    )
    .join(dim_operator_df, on="operator_name", how="left")
    .join(dim_currency_df, on="currency", how="left")
    .select(
        "date",
        "tracking_id",
        "operator_id",
        "registrations",
        "ftds",
        "total_commission",
        "currency_id",
        "card_id",
        "campaign_id"
    )
)

# ==========================================================
# 2. LOAD AFFMATRIX DATA
# ==========================================================
affmatrix_src = spark.read.table("LH_PeakMedia.dbo.affmatrix_clean")

affmatrix_base = (
    affmatrix_src
    .withColumn("tracking_id", F.lpad(F.col("tracking_id").cast("string"), 12, "0"))
    .withColumn("registration_date", F.to_date(F.col("registration_date")))
    .withColumn("ftd_date", F.to_date(F.col("ftd_date")))
    .withColumn("commission_earned", F.col("commission_earned").cast("decimal(18,2)"))
    .withColumn("currency", F.trim(F.col("currency")))
    .withColumn("operator_name", F.trim(F.col("operator_name")))
    .withColumn("card_id", F.substring(F.col("tracking_id"), 7, 3).cast("int"))
    .withColumn("campaign_id", F.substring(F.col("tracking_id"), 10, 3))
)

# ==========================================================
# 3. REGISTRATIONS AGGREGATION
# ==========================================================
affmatrix_reg_df = (
    affmatrix_base
    .filter(F.col("registration_date").isNotNull())
    .groupBy(
        F.col("registration_date").alias("date"),
        F.col("tracking_id"),
        F.col("operator_name"),
        F.col("currency"),
        F.col("card_id"),
        F.col("campaign_id")
    )
    .agg(
        F.count(F.col("registration_date")).cast("int").alias("registrations")
    )
    .withColumn("ftds", F.lit(0).cast("int"))
    .withColumn("total_commission", F.lit(0).cast("decimal(18,2)"))
)

# ==========================================================
# 4. FTDS + COMMISSION AGGREGATION
# ==========================================================
affmatrix_ftd_df = (
    affmatrix_base
    .filter(F.col("ftd_date").isNotNull())
    .groupBy(
        F.col("ftd_date").alias("date"),
        F.col("tracking_id"),
        F.col("operator_name"),
        F.col("currency"),
        F.col("card_id"),
        F.col("campaign_id")
    )
    .agg(
        F.count(F.col("ftd_date")).cast("int").alias("ftds"),
        F.sum(F.coalesce(F.col("commission_earned"), F.lit(0))).cast("decimal(18,2)").alias("total_commission")
    )
    .withColumn("registrations", F.lit(0).cast("int"))
)

# ==========================================================
# 5. COMBINE AFFMATRIX REG + FTD
# ==========================================================
affmatrix_df = (
    affmatrix_reg_df
    .unionByName(affmatrix_ftd_df)
    .groupBy(
        "date",
        "tracking_id",
        "operator_name",
        "currency",
        "card_id",
        "campaign_id"
    )
    .agg(
        F.sum("registrations").cast("int").alias("registrations"),
        F.sum("ftds").cast("int").alias("ftds"),
        F.sum("total_commission").cast("decimal(18,2)").alias("total_commission")
    )
    .join(dim_operator_df, on="operator_name", how="left")
    .join(dim_currency_df, on="currency", how="left")
    .select(
        "date",
        "tracking_id",
        "operator_id",
        "registrations",
        "ftds",
        "total_commission",
        "currency_id",
        "card_id",
        "campaign_id"
    )
)

# ==========================================================
# 6. APPEND AFFMATRIX DATA TO TRACKAFF DATA
# ==========================================================
final_df = trackaff_df.unionByName(affmatrix_df)

final_df = (
    final_df
    .groupBy(
        "date",
        "tracking_id",
        "operator_id",
        "currency_id",
        "card_id",
        "campaign_id"
    )
    .agg(
        F.sum("registrations").cast("int").alias("registrations"),
        F.sum("ftds").cast("int").alias("ftds"),
        F.sum("total_commission").cast("decimal(18,2)").alias("total_commission")
    )
    .select(
        "date",
        "tracking_id",
        "operator_id",
        "registrations",
        "ftds",
        "total_commission",
        "currency_id",
        "card_id",
        "campaign_id"
    )
)

# ==========================================================
# 7. WRITE TO LAKEHOUSE
# ==========================================================
final_df.write.mode("overwrite").saveAsTable("LH_PeakMedia.dbo.fact_affiliate_performance")

StatementMeta(, 86a77efb-2b6a-46be-8ee1-b2b5229fa855, 3, Finished, Available, Finished, False)

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("LoadFactCampaignSpend").getOrCreate()

# ==========================================================
# 1. READ SOURCE
# ==========================================================
src = spark.read.table("LH_PeakMedia.dbo.campaign_clean")

# ==========================================================
# 2. TRANSFORM
# ==========================================================
df = (
    src.select(
        F.to_date(F.col("date"), "dd/MM/yyyy").alias("date"),
        F.lpad(F.col("campaign_id").cast("string"), 3, "0").alias("campaign_id"),
        F.col("impressions").cast("int").alias("impressions"),
        F.col("clicks").cast("int").alias("clicks"),
        F.col("inbounds").cast("int").alias("inbounds"),
        F.col("outbounds").cast("int").alias("outbounds"),
        F.col("spend_gbp").cast("decimal(18,2)").alias("spend_gbp")
    )
)

# ==========================================================
# 3. WRITE TO TABLE
# ==========================================================
df.write.mode("overwrite").saveAsTable("LH_PeakMedia.dbo.fact_campaign_spend")

StatementMeta(, 463624e1-2142-45ad-a84c-e6a7f63d3f87, 6, Finished, Available, Finished, False)

Calling getAccessToken from PyTridentTokenLibrary


SynapseWidget(Synapse.DataFrame, 89346598-7b22-4a1c-8758-d1b2d9d36c48)